In [1]:
import wfdb
import numpy as np
import pandas as pd
import time
import os
import json

VITAL = "HR"
DATA_PATH = f"../data/pretrain_vitals_data/{VITAL}"
os.makedirs(DATA_PATH, exist_ok=True)

TARGET_VITALS = [VITAL]

# Identify Matched Record IDs

In [46]:
import wfdb
import pandas as pd
from wfdb.io import download

matched_records = wfdb.get_record_list('mimic3wdb-matched')

In [50]:
matched_record_ids = set()

print(f"Parsing {len(matched_records)} patients...")

for i, patient_dir in enumerate(matched_records):  # e.g. 'p00/p000020/'
    if i % 100 == 0 and i != 0:
        print(f"    {i} patients parsed, {len(matched_record_ids)} records found")
    patient_dir_clean = patient_dir.strip('/')
    try:
        content = download._stream_header('RECORDS', f'mimic3wdb-matched/1.0/{patient_dir_clean}')
        for line in content.strip().split('\n'):
            line = line.strip()
            # We want lines like '3544749_0001', extract base record id '3544749'
            if line and line[0].isdigit():
                record_id = line.split('_')[0]  # '3544749_0001' -> '3544749'
                matched_record_ids.add(int(record_id))
    except Exception as e:
        print(f"Failed for {patient_dir_clean}: {e}")

print(f"Unique numeric record IDs in matched subset: {len(matched_record_ids)}")
print(list(matched_record_ids)[:10])

Parsing 10282 patients...
    100 patients parsed, 187 records found
    200 patients parsed, 369 records found
    300 patients parsed, 571 records found
    400 patients parsed, 760 records found
    500 patients parsed, 945 records found
    600 patients parsed, 1142 records found
    700 patients parsed, 1336 records found
    800 patients parsed, 1528 records found
    900 patients parsed, 1744 records found
    1000 patients parsed, 1958 records found
    1100 patients parsed, 2157 records found
    1200 patients parsed, 2343 records found
    1300 patients parsed, 2523 records found
    1400 patients parsed, 2733 records found
    1500 patients parsed, 2943 records found
    1600 patients parsed, 3160 records found
    1700 patients parsed, 3364 records found
    1800 patients parsed, 3546 records found
    1900 patients parsed, 3746 records found
    2000 patients parsed, 3938 records found
    2100 patients parsed, 4137 records found
    2200 patients parsed, 4352 records foun

In [51]:
import json
with open('matched_record_ids.json', 'w') as f:
    json.dump(list(matched_record_ids), f)

print(f"Saved {len(matched_record_ids)} record IDs to matched_record_ids.json")

Saved 22242 record IDs to matched_record_ids.json


# Ingest Unmatched Records

In [2]:
import requests

print("Fetching master record paths from PhysioNet...")
full_db_url = "https://physionet.org/files/mimic3wdb/1.0/RECORDS-numerics"
all_full_paths = requests.get(full_db_url).text.splitlines()

path_map = {}
for p in all_full_paths:
    top = int(p.split('/')[0])
    record = p.split('/')[1]
    if top not in path_map:
        path_map[top] = []
    path_map[top].append(record)

Fetching master record paths from PhysioNet...


In [6]:
with open('matched_record_ids.json', 'r') as f:
    matched_record_ids = set(json.load(f))
len(matched_record_ids)

top_dir = [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]

for t in top_dir:
    CHECKPOINT_PATH = f"{DATA_PATH}/unmatched_checkpoint_{VITAL}_{t}.json"
    OUTPUT_PATH = f"{DATA_PATH}/unmatched_mimic_vitals{VITAL}_{t}.parquet"

    print(f"Exploring folder {t}...")

    # Load checkpoint if exists
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            checkpoint = json.load(f)
        processed_records = set(checkpoint["processed_records"])
        record_hit_rate    = checkpoint["record_hit_rate"]
        missing_vitals     = checkpoint["missing_vitals"]
        records           = checkpoint["records"]
        print(f"Resuming from checkpoint: {len(processed_records)} records already processed")
    else:
        processed_records = set()
        record_hit_rate    = {"total": 0, "keep": 0, "missing": 0, "short": 0}
        missing_vitals     = {v: 0 for v in TARGET_VITALS}
        records           = 0
        print("Starting fresh")
    
    if os.path.exists(OUTPUT_PATH):
        df = pd.read_parquet(OUTPUT_PATH)
        print(f"  Loaded existing data: {len(df):,} rows")
    else:
        df = None

    records_since_save = 0

    sub_dir = path_map[t]
    
    for record_id in sub_dir:        
        if record_id in processed_records:
            continue
        
        if record_id in matched_record_ids:
            print(f"     {record_id} found in matched subset, skipping...")
            
        else:
            try:
                # 1. Read Header
                header = wfdb.rdheader(
                    f"{record_id}n",
                    pn_dir=f"mimic3wdb/1.0/{t}/{record_id}"
                )
                
                print(f"     Reading {record_id}")
                records += 1
                record_hit_rate["total"] += 1
                
                # 2. Check Vitals
                if header.sig_name is None or not all(v in header.sig_name for v in TARGET_VITALS):
                    record_hit_rate["missing"] += 1
                    for vital in TARGET_VITALS:
                        if vital not in (header.sig_name or []):
                            missing_vitals[vital] += 1
                    print("       Skipping: Missing vitals...")
                
                else:
                    # 3. Download & Process Data
                    max_samples = min(60 * (1440 + 10), header.sig_len)
                    t0 = time.time()
                    record = wfdb.rdrecord(
                        f"{record_id}n",
                        pn_dir=f"mimic3wdb/1.0/{t}/{record_id}",
                        channel_names=TARGET_VITALS,
                        physical=True,
                        sampfrom=0,
                        sampto=max_samples
                    )
                    print(f"       Downloaded in {time.time() - t0:.1f} seconds")

                    rec_df = pd.DataFrame(record.p_signal, columns=record.sig_name)
                    rec_df = rec_df.reindex(columns=TARGET_VITALS)

                    fs = record.fs
                    samples_per_minute = round(fs * 60)
                    rec_df["Minute"] = np.arange(len(rec_df)) // samples_per_minute

                    max_minute = rec_df["Minute"].max()
                    rec_df = rec_df[(rec_df["Minute"] >= 5) & (rec_df["Minute"] <= max_minute - 5)]

                    if samples_per_minute > 1:
                        rec_df = rec_df.groupby("Minute")[TARGET_VITALS].mean().reset_index()

                    if rec_df["Minute"].nunique() < 120:
                        print(f"         Skipping: less than 2 hours of data")
                        record_hit_rate["short"] += 1
                    else:
                        rec_df = rec_df[rec_df["Minute"] < (rec_df["Minute"].min() + 1440)]
                        rec_df["Timestamp"] = pd.to_timedelta(rec_df["Minute"], unit="min")
                        rec_df["Record"] = record_id
                        record_hit_rate["keep"] += 1

                        print(f"         {len(rec_df)} rows")

                        if df is None:
                            df = rec_df.reset_index(drop=True)
                        else:
                            df = pd.concat([df, rec_df.reset_index(drop=True)])
                            
            except Exception as e:
                # File not found or read error
                pass 


        processed_records.add(record_id)
        records_since_save += 1

        # Checkpoint Saving Logic
        if records_since_save >= 100:
            if df is not None:
                df = df.reindex(columns=["Record", "Minute"] + TARGET_VITALS).reset_index(drop=True)
                df.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
                print(f"Saved {len(df):,} rows to {OUTPUT_PATH}")
            
            with open(CHECKPOINT_PATH, "w") as f:
                json.dump({
                    "processed_records": list(processed_records),
                    "record_hit_rate":    record_hit_rate,
                    "missing_vitals":     missing_vitals,
                    "records":            records
                }, f)
            print(f"  Checkpoint saved: {len(processed_records)} records explored")
            records_since_save = 0

    # Final save at the end of the directory
    if df is not None:
        df = df.reindex(columns=["Record", "Minute"] + TARGET_VITALS).reset_index(drop=True)
        df.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
        print(f"Final Save {len(df):,} rows to {OUTPUT_PATH}")

    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({
            "processed_records": list(processed_records),
            "record_hit_rate":    record_hit_rate,
            "missing_vitals":     missing_vitals,
            "records":            records
        }, f)
    
    print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")
    print(f"Records Found: {records}")
    print(f"Record hit rate:")
    for k, v in record_hit_rate.items():
        print(f"  {k}: {v}")

Exploring folder 30...
Starting fresh
     Reading 3000003
       Downloaded in 1.0 seconds
         1440 rows
     Reading 3000031
       Downloaded in 0.2 seconds
         Skipping: less than 2 hours of data
     Reading 3000060
       Downloaded in 4.2 seconds
         1440 rows
     Reading 3000063
       Downloaded in 4.7 seconds
         1440 rows
     Reading 3000065
       Downloaded in 0.1 seconds
         Skipping: less than 2 hours of data
     Reading 3000086
       Downloaded in 1.6 seconds
         Skipping: less than 2 hours of data
     Reading 3000100
       Downloaded in 3.1 seconds
         164 rows
     Reading 3000103
       Downloaded in 1.2 seconds
         1003 rows
     Reading 3000105
       Downloaded in 0.3 seconds
         337 rows
     Reading 3000125
       Downloaded in 11.8 seconds
         1440 rows
     Reading 3000126
       Downloaded in 1.4 seconds
         1440 rows
     Reading 3000142
       Downloaded in 4.2 seconds
         1440 rows
     Read

KeyboardInterrupt: 